### 구현할 기능 목록
#### GMAIL
* 메일 확인 (기간 & 키워드 조건)
* 메일 전송 기능
* (+ 메일 템플릿 추천 -> 에이전트와 결합할 때, LLM Agent 연결을 통해 구현?)
* (+ Query 프롬프트로부터 각 메서드의 인수들을, LLM Agent를 통해 적절한 type으로 받을 수 있도록 )

#### Calendar
* 일정 확인 (기간 & 키워드 조건)
* 새로운 일정 추가
* 기존 일정 수정

In [6]:
# 라이브러리 import

import os.path
import base64
import datetime
from bs4 import BeautifulSoup
import mimetypes

# Google 라이브러리
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError

# 이메일 작성을 위한 라이브러리
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders

# KST
KST = datetime.timezone(datetime.timedelta(hours=9))

In [42]:
# Gmail(읽기, 수정, 전송 권한) + Calendar(전체 권한)
SCOPES = [
    "https://www.googleapis.com/auth/gmail.modify",
    "https://www.googleapis.com/auth/calendar"
]

In [ ]:
# Google API 서비스 인증


def get_gamail_service():
    '''
    Google Gmail API 서비스 객체를 인증 및 반환.
    token.json이 경로 내에 존재하지 않으면 새로 생성.
    '''

    creds = None

    # 'token.json' : 인증 정보를 저장
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)

    # 토큰이 없거나 유효하지 않은 경우, 사용자가 직접 로그인하도록 함
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'google_oauth_credentials.json', SCOPES
            )
            creds = flow.run_local_server(port=0)

    # 다음 번 실행을 위해 토큰을 저장
    with open('token.json', 'w') as token:
        token.write(creds.to_json())

    # API 서비스 빌드
    try:
        service = build('gmail', 'v1', credentials=creds)
        
        # 로그
        print("Gmail 서비스 인증 성공")
        
        return service
    
    except HttpError as e:
        # 로그
        print(f"서비스 빌드 실패 : {e}")

        return None


def list_emails_by_keyword_and_date(service, start_date, end_date, keyword, message_num = 10):
    '''
    'start_date' ~ 'end_date'에 수신한 메일 중 메일 '제목' or '내용'에 특정 키워드가 포함된 메일 제목 확인
    '''

    email_list = []

    try:
        # 1.1. KST 기준 시작일, 종료일 설정
        date_start_kst = datetime.datetime.strptime(start_date, '%Y-%m-%d').replace(tzinfo=KST)
        date_end_kst =  datetime.datetime.strptime(end_date, '%Y-%m-%d').replace(tzinfo=KST) + datetime.timedelta(days=1)

        # 1.2. Unix 타임스탬프로 변환
        st_timestamp = int(date_start_kst.timestamp())
        end_timestamp = int(date_end_kst.timestamp())

        # 2. Gmail 검색 쿼리
        query = f"(subject:({keyword}) OR body:({keyword})) -in:sent after:{st_timestamp} before:{end_timestamp}"

        # 3. 메일 검색
        results = service.users().messages().list(userId='me', q=query).execute()
        messages = results.get('messages', [])

        if not messages:
            print("해당 키워드를 포함하는 메일이 존재하지 않음.")
            return []
        
        # 4. 각 메일 추출
        for msg in messages[:message_num]:
            msg_data = service.users().messages().get(
                userId='me', id=msg['id'], format='full'
            ).execute()

            # 4.1. 본문 추출
            payload = msg_data['payload']
            body_data = ""
            decoded_body = "본문 없음"

            # HTML, TEXT 등 여러 part로 나뉜 경우
            if 'parts' in payload:
                for part in payload['parts']:
                    if part['mimeType'] == 'text/plain':
                        body_data = part['body'].get('data', '')
                        break
                        
                    elif part['mimeType'] == 'text/html':
                        body_data = part['body'].get('data', '')

            # 단순 텍스트 메일인 경우
            elif 'body' in payload:
                body_data = payload['body'].get('data', '')

            if body_data:
                try:
                    decoded_html = base64.urlsafe_b64decode(body_data.encode('ASCII')).decode('utf-8')

                    soup = BeautifulSoup(decoded_html, 'html.parser')
                    decoded_body = soup.get_text(separator=' ', strip=True)
                
                except Exception as e:
                    decoded_body = f"본문 파싱 실패 : {e}"

            # 4.2. 제목 및 발신인 추출
            headers = payload['headers']

            subject = next((h['value'] for h in headers if h['name'] == 'Subject'), "제목 없음")
            sender = next((h['value'] for h in headers if h['name'] == 'From'), "발신인 불명")

            # 4.3. Gmail 링크 생성
            mail_id = msg['id']
            mail_link = f"https://mail.google.com/mail/u/0/#inbox/{mail_id}" # gmail 계정이 여러 개일 경우 계정의 순서에 맞게 0부터 순차적으로 증가하는 적절한 숫자를 '.../u/{숫자}/#inbox/...'의 {숫자}에 삽입하면 됨!
            
            email_list.append([subject, sender, decoded_body, mail_link])

        return email_list

    except HttpError as e:
        print(f"키워드 기준 메일 제목 확인 실패 : {e}")

        return []
    
def send_email(service, to, subject, body_text, file_path=None):
    '''
    지정된 주소로 메일 전송 (첨부파일 포함 기능 O)

    Args:
        service : Gmail API 서비스 객체
        to (str) : 수신자 이메일 주소
        subject (str) : 메일 제목
        body_text (str) : 메일 본문 (일반 텍스트)
        file_path (str, optional) : 첨부할 파일의 경로 (default=None)
    '''

    try:
        # 1. 메일의 기본 틀(MIMEMultipart) 생성
        message = MIMEMultipart()
        message['to'] = to
        message['subject'] = subject

        # 2. 메일 본문(MIMEText) 추가
        msg_body = MIMEText(body_text, "plain")
        message.attach(msg_body)

        # 3. 첨부파일 처리
        if file_path and os.path.exists(file_path):
            # 파일의 type 추측
            content_type, encoding = mimetypes.guess_type(file_path)

            main_type, sub_type = content_type.split("/", 1)

            # 파일 읽기
            with open(file_path, 'rb')as fp:
                file_data = fp.read()

            # MIMEBase 객체 생성 및 Base64 인코딩
            attachment = MIMEBase(main_type, sub_type)
            attachment.set_payload(file_data)
            encoders.encode_base64(attachment)

            # 첨부파일 헤더 추가
            file_name = os.path.basename(file_path)
            attachment.add_header(
                'content-disposition',
                'attachment',
                filename=file_name
            )

            # 메일 객체에 첨부파일 추가
            message.attach(attachment)

        # 4. 최종 메일 객체를 Base64 인코딩
        raw_message = base64.urlsafe_b64encode(message.as_bytes()).decode('utf-8')

        # 5. Gmail API로 전송
        send_data = {'raw': raw_message}
        service.users().messages().send(userId='me', body=send_data).execute()

    except FileNotFoundError:
        print(f"첨부 파일 경로 찾기 실패. 경로 : {file_path}")

    except HttpError as e1:
        print(f"메일 전송 실패 : {e1}")

    except Exception as e2:
        print(f"메일 전송 실패 : {e2}")

In [12]:
gmail_service = get_gamail_service()

Gmail 서비스 인증 성공


In [62]:
results = list_emails_by_keyword_and_date(gmail_service, '2025-09-16', '2025-09-16', '출석', 5)

In [63]:
for 제목, 발신인, 본문, 링크 in results:
    print(f"Title: {제목}")
    print(f"From : {발신인}")
    print(f"Text : \n{본문}")
    print(f"GMAIL Link : {링크}")
    print("-"*100)

Title: Re: [조합최적화(001) 강좌 출석 관련 문의]
From : "노영주" <youngjooroh@snu.ac.kr>
Text : 
안녕하세요, 강의조교 노영주 입니다.

보내주신 서류 잘 확인했습니다.
해당 일자는 출석인정 처리하도록 하겠습니다.

감사합니다.

노영주 드림

*Youngjoo Roh*
Ph.D. Student
Systems Optimization Lab.
Department of Industrial Engineering, Seoul National University
*Email* youngjooroh@snu.ac.kr | youngjooroh.yjr@gmail.com


2025년 9월 16일 (화) 오후 12:20, ­임다움 / 학생 / 산업공학과 님이 작성:

> 안녕하세요, 조교님.
> 홍성필 교수님의 조합최적화(001) 강좌를 수강하고 있는 산업공학과에 재학 중인 임다움입니다.
>
> 다름이 아니라, 비염 및 편도염 증세가 심해져 병원에 방문하게 되어 부득이하게 금일 09.16(화)에 진행되었던 강의에 출석하지
> 못했습니다. 병원에서 처방전을 발급해주어 첨부하오니, 이를 통해 출석을 인정 받을 수 있는지 문의드리고자 메일 드립니다. 혹시, 이외의
> 증명을 위한 서류가 필요한 경우 말씀해주시면 추후 제출하도록 하겠습니다.
>
> 감사합니다.
>
> 임다움 드림
>
GMAIL Link : https://mail.google.com/mail/u/1/#inbox/19950cfa1ca452fb
----------------------------------------------------------------------------------------------------


In [13]:
content = "Gmail API 활용 메일 전송 기능 test 중.\n쿼카 사진 첨부 파일도 정상적으로 첨부되었는지 확인 바람.\n굿!"

send_email(gmail_service, 'dlaekdna4862@gmail.com', '메일 전송 기능 test', content, './quokka.jpeg')

In [14]:
send_email(gmail_service, '0101yhchoi@gmail.com', '안뇽~', '안뇽~', './quokka.jpeg')

In [50]:
# Google Calendar API 서비스 인증
def get_calendar_service():
    '''
    Google Calendar API 서비스 객체 인증 및 반환
    token.json이 존재하지 않은 경우 새로 생성
    '''
    
    creds = None
    
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'google_oauth_credentials.json', SCOPES
            )
            creds = flow.run_local_server(port=0)

    with open('token.json', 'w') as token:
        token.write(creds.to_json())

    try:
        service = build('calendar', 'v3', credentials=creds)
        
        # 로그
        print("Google Calendar 서비스 인증 성공")
        
        return service
    
    except HttpError as e:
        # 로그
        print(f"서비스 빌드 실패 : {e}")

        return None
    
# 일정 확인 (기간 & 키워드 조건)
def list_events(service, start_date, end_date, calendarId='primary', keyword=None):
    """
    특정 기간(YYYY-MM-DD) 사이의 일정을 검색합니다.
    키워드가 주어지면 해당 키워드가 포함된 일정만 필터링합니다.
    """

    try:
        # KST 기준으로 날짜를 datetime 객체로 변환
        start_dt = datetime.datetime.strptime(start_date, "%Y-%m-%d").replace(tzinfo=KST)
        # end_date 당일을 포함하기 위해 1일을 더함
        end_dt = datetime.datetime.strptime(end_date, "%Y-%m-%d").replace(tzinfo=KST) + datetime.timedelta(days=1)

        # Google API가 요구하는 RFC3339(ISO 8601) 형식으로 변환
        start_iso = start_dt.isoformat()
        end_iso = end_dt.isoformat()

        events_result = (
            service.events()
            .list(
                calendarId=calendarId, # primary는 기본 캘린더
                timeMin=start_iso,     # 시작 시간
                timeMax=end_iso,       # 종료 시간
                q=keyword,             # 키워드 검색
                maxResults=10,
                singleEvents=True,
                orderBy="startTime",
            )
            .execute()
        )

        events = events_result.get("items", [])

        return events
            
    except HttpError as e:
        print(f"구글 캘린더 일정 확인 실패 : {e}")

# 새로운 일정 추가
def create_event(service, title, start_str, end_str, description, overrides, location=None, attendees=None):
    """
    새로운 일정 추가

    datetime_str 형식: "YYYY-MM-DD" or "YYYY-MM-DD HH:MM" (예: "2025-11-10 14:00")
    overrides = [{"method":(str), "minutes": (int)}]
    attendees: ["email1@example.com", "email2@example.com"]
    """

    event_body = {
            'summary': title,
            'description' : description,
            'reminders' : {
                'useDefault': False,
                'overrides': [{'method': method, 'minutes': minutes} for method, minutes in overrides]
            },
            'location': location,
            'attendees': [{"email": email} for email in (attendees or [])]
    }

    try:
        # 1. 시간 지정 일정 (예: "2025-11-10 14:00")
        if len(start_str) > 10: # 10글자("YYYY-MM-DD")보다 길면 시간으로 간주
            start_dt = datetime.datetime.strptime(start_str, "%Y-%m-%d %H:%M").replace(tzinfo=KST)
            end_dt = datetime.datetime.strptime(end_str, "%Y-%m-%d %H:%M").replace(tzinfo=KST)
            
            event_body["start"] = {
                "dateTime": start_dt.isoformat(), # 'dateTime' 사용
                "timeZone": "Asia/Seoul",
            }

            event_body["end"] = {
                "dateTime": end_dt.isoformat(), # 'dateTime' 사용
                "timeZone": "Asia/Seoul",
            }

        # 2. 종일 일정 (예: "2025-11-10")
        else:
            start_date_obj = datetime.datetime.strptime(start_str, "%Y-%m-%d").date()
            
            # [중요] 종료일(end_str) 당일을 포함하기 위해 +1일을 더함
            end_date_obj = datetime.datetime.strptime(end_str, "%Y-%m-%d").date()
            end_date_exclusive = end_date_obj + datetime.timedelta(days=1)
            
            event_body["start"] = {
                "date": start_date_obj.isoformat() # 'date' 사용
            }
            event_body["end"] = {
                "date": end_date_exclusive.isoformat() # 'date' 사용
            }

        created_event = (
            service.events()
            .insert(calendarId="primary", body=event_body)
            .execute()
        )
        
        print(f"✅ 일정 생성 성공!")
        print(f"링크: {created_event.get('htmlLink')}")

    except HttpError as e:
        print(f"구글 캘린더 일정 추가 실패 : {e}")

# 기존 일정 수정 ---
def modify_event(service, event_id, calendarId='primary', new_title=None, new_discription=None, new_start_datetime_str=None, new_end_datetime_str=None):
    """
    기존 일정의 내용을 수정 (event_id 필요)
    event_id는 list_events() 메서드를 통해 얻을 수 있음
    """

    if not any([new_title, new_start_datetime_str, new_end_datetime_str]):
        print("수정할 내용이 없습니다.")
        return
        
    try:
        # 1. 수정을 위해 기존 이벤트 정보를 가져옵니다. (필수)
        event = service.events().get(calendarId=calendarId, eventId=event_id).execute()

        # 2. 변경할 내용만 업데이트합니다.
        if new_title:
            event["summary"] = new_title

        if new_discription:
            event["description"] = new_discription

        if new_start_datetime_str:
            start_dt = datetime.datetime.strptime(new_start_datetime_str, "%Y-%m-%d %H:%M").replace(tzinfo=KST)
            event["start"] = {"dateTime": start_dt.isoformat(), "timeZone": "Asia/Seoul"}
        
        if new_end_datetime_str:
            end_dt = datetime.datetime.strptime(new_end_datetime_str, "%Y-%m-%d %H:%M").replace(tzinfo=KST)
            event["end"] = {"dateTime": end_dt.isoformat(), "timeZone": "Asia/Seoul"}

        # 3. 'patch' 또는 'update'로 이벤트를 수정합니다. (patch는 변경된 필드만 업데이트)
        updated_event = (
            service.events()
            .update(calendarId=calendarId, eventId=event_id, body=event)
            .execute()
        )
        
        print(f"✅ 일정 수정 성공!")
        print(f"링크: {updated_event.get('htmlLink')}")

    except HttpError as error:
        print(f"기능 3 실패: {error} (Event ID를 확인하세요)")

# 기존 일정 삭제
def delete_event(service, event_id, calendarId='primary'):
    """
    기존 일정을 삭제합니다. (event_id가 필요합니다)
    event_id는 list_events() 메서드를 통해 얻을 수 있습니다.
    """

    try:
        service.events().delete(
            calendarId=calendarId, 
            eventId=event_id
        ).execute()
        
        print(f"✅ Event ID '{event_id}' 일정 삭제 성공!")

    except HttpError as error:
        print(f"기능 4 실패: {error} (Event ID를 확인하세요)")

In [43]:
calendar_service = get_calendar_service()

Google Calendar 서비스 인증 성공


In [25]:
events = list_events(calendar_service, '2025-12-15', '2025-12-15', '6001ae49331e09932a85e76651d4ee053258c4c7deb63c857fdc5b3862b234d9@group.calendar.google.com')

In [ ]:
# print(events)

for event in events:
    # 종일 일정이 아닌 경우
    if event['start'].get('dateTime'):
        st_iso_object = datetime.datetime.fromisoformat(event['start'].get('dateTime'))
        et_iso_object = datetime.datetime.fromisoformat(event['end'].get('dateTime'))

        st = st_iso_object.strftime("%Y년 %m월 %d일 %H시 %M분")
        et = et_iso_object.strftime("%Y년 %m월 %d일 %H시 %M분")

    # 종일 일정인 경우
    else:
        iso_object = datetime.datetime.fromisoformat(event['start'].get('date'))

        st = iso_object.strftime("%Y년 %m월 %d일 00시 00분")
        et = iso_object.strftime("%Y년 %m월 %d일 23시 59분")


    print(f"일정 : {event['summary']}")
    print(f"시간 : {st} ~ {et}")

    print("-"*200)

[{'kind': 'calendar#event', 'etag': '"3524992116146814"', 'id': '2ua7quhfnnlv616b5jppnbh6j1_20251215', 'status': 'confirmed', 'htmlLink': 'https://www.google.com/calendar/event?eid=MnVhN3F1aGZubmx2NjE2YjVqcHBuYmg2ajFfMjAyNTEyMTUgNjAwMWFlNDkzMzFlMDk5MzJhODVlNzY2NTFkNGVlMDUzMjU4YzRjN2RlYjYzYzg1N2ZkYzViMzg2MmIyMzRkOUBn', 'created': '2025-11-07T06:14:08.000Z', 'updated': '2025-11-07T06:14:27.057Z', 'summary': '삼성생명 보험료', 'description': '32,700원', 'creator': {'email': 'dlaekdna4862@gmail.com'}, 'organizer': {'email': '6001ae49331e09932a85e76651d4ee053258c4c7deb63c857fdc5b3862b234d9@group.calendar.google.com', 'displayName': '정기 결제', 'self': True}, 'start': {'date': '2025-12-15'}, 'end': {'date': '2025-12-16'}, 'recurringEventId': '2ua7quhfnnlv616b5jppnbh6j1', 'originalStartTime': {'date': '2025-12-15'}, 'transparency': 'transparent', 'iCalUID': '2ua7quhfnnlv616b5jppnbh6j1@google.com', 'sequence': 1, 'reminders': {'useDefault': False, 'overrides': [{'method': 'popup', 'minutes': 900}, {'meth

In [52]:
create_event(calendar_service, '일정 추가 test', '2025-11-07 16:46', '2025-11-07 17:00', '일정 추가 test 메모', [('popup', 30), ('email', 1440)])

✅ 일정 생성 성공!
링크: https://www.google.com/calendar/event?eid=dnVuaW1rOGlmN3BpZm43YjczYTA5ZjdjMXMgZGxhZWtkbmE0ODYyQG0


In [54]:
target_event_id = list_events(calendar_service, '2025-11-07', '2025-11-07', keyword='test')[0]['id']

In [49]:
modify_event(calendar_service, target_event_id, new_title='기존 일정 수정 기능 test', new_discription='기존 일정 수정 기능 test 메모', new_start_datetime_str='2025-11-07 17:06', new_end_datetime_str='2025-11-08 17:06')

✅ 일정 수정 성공!
링크: https://www.google.com/calendar/event?eid=cjFnNzc3ZXA4bnUwNWRtY2Jub2N2bmlqdTggZGxhZWtkbmE0ODYyQG0


In [55]:
delete_event(calendar_service, target_event_id)

✅ Event ID 'vunimk8if7pifn7b73a09f7c1s' 일정 삭제 성공!
